In [2]:
!pip install openai pinecone pypdf python-dotenv -q

In [7]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys are loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please create a .env file with your API key.")
if not os.getenv("PINECONE_API_KEY"):
    raise ValueError("PINECONE_API_KEY not found in environment variables. Please create a .env file with your API key.")

print("All API keys loaded successfully!")


All API keys loaded successfully!


In [11]:
from pypdf import PdfReader

pdf_path = "/Users/dinabosmabuczynska/Desktop/IRONHACK/Podcast Project/podcast.pdf"
reader = PdfReader(pdf_path)

text = ""
for page in reader.pages:
    text += page.extract_text()

print(f"Loaded {len(reader.pages)} pages, {len(text)} characters")
print(f"\nFirst 500 characters:\n{text[:500]}")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)


Loaded 15 pages, 30358 characters

First 500 characters:
How is ar)ﬁcial intelligence reshaping early childhood development?   Introduc*on: Over the past decades, ar*ﬁcial intelligence (AI) has transformed how we live, communicate or work.1 In spite of growing evidence of the beneﬁts of AI in healthcare, understanding of its impact on children’s cogni*ve, emo*onal, and behavioral development is s*ll limited.2,3 Given the profound importance of these early years in a person’s life, growth, success and well-being, it is essen*al to examine how AI can bo


In [13]:
def chunk_text(text, chunk_size=500, chunk_overlap=50):
    """Split text into chunks with overlap."""
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        chunks.append({
            "page_content": chunk,
            "metadata": {"start": start, "end": end}
        })

        start += chunk_size - chunk_overlap

        if chunk_size <= chunk_overlap:
            break

    return chunks

chunks = chunk_text(text, chunk_size=500, chunk_overlap=50)
print(f"Number of chunks: {len(chunks)}")
print(f"First chunk: {chunks[0]['page_content'][:100]}...")

Number of chunks: 68
First chunk: How is ar)ﬁcial intelligence reshaping early childhood development?   Introduc*on: Over the past dec...


In [15]:
from openai import OpenAI

# Initialize OpenAI client
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Embedding functions (replaces OpenAIEmbeddings)
def embed_query(text, client, model="text-embedding-3-small"):
    """Create an embedding for a single query text."""
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

def embed_documents(texts, client, model="text-embedding-3-small"):
    """Create embeddings for multiple texts (batch processing)."""
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [item.embedding for item in response.data]

# Create embeddings for a sample text
sample_text = "Machine learning enables systems to learn from data"
sample_embedding = embed_query(sample_text, openai_client)
print(f"Embedding dimension: {len(sample_embedding)}")
print(f"First 5 values: {sample_embedding[:5]}")

Embedding dimension: 1536
First 5 values: [-0.0168609619140625, -0.0004792213439941406, 0.020111083984375, -0.01496124267578125, 0.07781982421875]


In [18]:
from pinecone import Pinecone, ServerlessSpec

# Initialize Pinecone client
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Create or connect to an index
index_name = "rag-openai-index"

# Check if index exists, create if not
existing_indexes = [index.name for index in pc.list_indexes()]
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=1536,  # OpenAI text-embedding-3-small dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Created index: {index_name}")
else:
    print(f"Index {index_name} already exists")

Created index: rag-openai-index


In [19]:
# Connect to the index
index = pc.Index(index_name)

# Create embeddings for all chunks
chunk_texts = [chunk["page_content"] for chunk in chunks]
chunk_embeddings = embed_documents(chunk_texts, openai_client)

# Prepare vectors for upsert
vectors_to_upsert = []
for i, (chunk, embedding) in enumerate(zip(chunks, chunk_embeddings)):
    vectors_to_upsert.append({
        "id": f"chunk-{i}",
        "values": embedding,
        "metadata": {
            "text": chunk["page_content"],
            **chunk["metadata"]
        }
    })

# Upsert vectors to Pinecone
index.upsert(vectors=vectors_to_upsert)
print(f"Added {len(vectors_to_upsert)} documents to Pinecone vector store")

/Users/dinabosmabuczynska/Desktop/bootcamp_env/.conda/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Added 68 documents to Pinecone vector store


In [21]:
# Perform a similarity search
def similarity_search(query, index, openai_client, k=2):
    """Search for similar documents using query embedding."""
    # Embed the query
    query_embedding = embed_query(query, openai_client)

    # Query Pinecone
    results = index.query(
        vector=query_embedding,
        top_k=k,
        include_metadata=True
    )

    # Format results
    documents = []
    for match in results.matches:
        documents.append({
            "page_content": match.metadata["text"],
            "metadata": {k: v for k, v in match.metadata.items() if k != "text"}
        })

    return documents

# Perform a search
query = "How is AI affecting early childhood development?"
results = similarity_search(query, index, openai_client, k=2)

print(f"Query: {query}")
print(f"\nRetrieved {len(results)} documents:")
for i, doc in enumerate(results, 1):
    print(f"\n{i}. {doc['page_content']}")

Query: How is AI affecting early childhood development?

Retrieved 2 documents:

1. How is ar)ﬁcial intelligence reshaping early childhood development?   Introduc*on: Over the past decades, ar*ﬁcial intelligence (AI) has transformed how we live, communicate or work.1 In spite of growing evidence of the beneﬁts of AI in healthcare, understanding of its impact on children’s cogni*ve, emo*onal, and behavioral development is s*ll limited.2,3 Given the profound importance of these early years in a person’s life, growth, success and well-being, it is essen*al to examine how AI can bo

2. echnology industries, policy-makers, and caregivers.5 But how much do we know? AI, a lever for early childhood development: Limited research has been conducted on the impact of AI on young children’s development. The few studies available so far have showed that AI oﬀers exci*ng opportuni*es for early childhood development (ECD) that can both enhance learning and support caregivers. Integra*ng AI technologie